# FunctionGemma 270M × TinyCeNN Memory Fusion — Sequential Acceptance

Uses `vtava/functiongemma-270m-it-simple-tool-calling`. Only Gemma3 full-attention anchors are replaced; sliding-window layers stay unchanged. Training is resumable on Drive, backed up privately during execution, and the accepted/current checkpoint is explicitly published to Hugging Face.

In [ ]:
import os, sys, subprocess, shutil
from pathlib import Path
assert subprocess.run(["nvidia-smi"], check=False).returncode == 0, "Enable GPU runtime"
REPO=Path("/content/TinyCeNN-LM")
if REPO.exists(): shutil.rmtree(REPO)
subprocess.run(["git","clone","--depth","1","https://github.com/vtavakkoli/TinyCeNN-LM.git",str(REPO)],check=True)
subprocess.run([sys.executable,"-m","pip","install","-q","transformers==4.57.6","datasets>=3,<5","huggingface_hub>=0.36","pytest","pandas"],check=True)
subprocess.run([sys.executable,"-m","pip","install","-q","-e",str(REPO),"--no-deps"],check=True)
for p in (str(REPO),str(REPO/"src")):
    if p not in sys.path: sys.path.insert(0,p)
os.environ["PYTHONPATH"]=os.pathsep.join([str(REPO),str(REPO/"src")])
import tinycenn_lm
print("✅ repo",subprocess.check_output(["git","-C",str(REPO),"rev-parse","HEAD"],text=True).strip())

In [ ]:
import json, torch
from huggingface_hub import HfApi
from transformers import AutoConfig
from google.colab import drive
MODEL_ID="vtava/functiongemma-270m-it-simple-tool-calling"
MODEL_REVISION=HfApi().model_info(MODEL_ID).sha
FEATURE_DIM=32; MEMORY_RANK=64; CONTEXT=128; MAX_ROUNDS_PER_RUN=4
RESET_PROGRESS=False
HF_MODEL_REPO="vtava/functiongemma-270m-it-simple-tool-calling-memory-fusion"
HF_PRIVATE=False; PUBLISH_TO_HF=True
drive.mount("/content/drive")
OUT=Path("/content/drive/MyDrive/TinyCeNN-LM/functiongemma-memory-fusion-sequential-r64")
if RESET_PROGRESS and OUT.exists(): shutil.rmtree(OUT)
OUT.mkdir(parents=True,exist_ok=True); LOG=OUT/"last_colab_run.log"
cfg=AutoConfig.from_pretrained(MODEL_ID,revision=MODEL_REVISION).get_text_config(decoder=True)
full=[i for i,k in enumerate(cfg.layer_types) if k=="full_attention"]
print({"revision":MODEL_REVISION,"full_attention":full,"output":str(OUT),"hf_repo":HF_MODEL_REPO})

## Hugging Face login
Add a Hugging Face **write** token to Colab Secrets as `HF_TOKEN`. This is required for both live backup and model publication.

In [ ]:
from huggingface_hub import HfApi, login, notebook_login
from google.colab import userdata
try: token=userdata.get("HF_TOKEN")
except Exception: token=None
if token: login(token=token,add_to_git_credential=False)
else: notebook_login()
print("✅ HF user",HfApi().whoami().get("name"))

In [ ]:
env=dict(os.environ,CUDA_VISIBLE_DEVICES="",OMP_NUM_THREADS="1",MKL_NUM_THREADS="1")
r=subprocess.run([sys.executable,"-m","pytest","-q","tests/test_gemma3_memory_fusion.py"],cwd=REPO,env=env,text=True,stdout=subprocess.PIPE,stderr=subprocess.STDOUT)
print(r.stdout)
if r.returncode: raise RuntimeError(f"preflight failed: {r.returncode}")
print("✅ preflight passed")

## Train / resume
The trainer is called directly (no `bash | tee`). This lets TinyCeNN recognize the `train_*.py` process, stream it, and perform mandatory Hugging Face backup correctly.

In [ ]:
cmd=[sys.executable,"-u",str(REPO/"scripts"/"train_functiongemma_memory_fusion_sequential.py"),
 "--base-model",MODEL_ID,"--model-revision",MODEL_REVISION,"--output-dir",str(OUT),
 "--feature-dim",str(FEATURE_DIM),"--memory-rank",str(MEMORY_RANK),"--context-length",str(CONTEXT),"--probe-context",str(CONTEXT),
 "--seed","73","--min-layer-steps","50","--max-layer-steps","300","--check-every","25","--layer-lr","0.0002",
 "--teacher-alpha-start","0.9","--teacher-alpha-end","0.0","--accept-nmse","0.2","--accept-cosine","0.9",
 "--accept-incremental-delta-nll","0.015","--accept-cumulative-delta-nll","0.05","--max-runtime-minutes","240","--resume","--strict-acceptance"]
run_env=dict(os.environ); run_env["SEQUENTIAL_MAX_ROUNDS_PER_RUN"]=str(MAX_ROUNDS_PER_RUN); run_env["PYTHONPATH"]=os.environ["PYTHONPATH"]
print(" ".join(cmd),flush=True)
result=subprocess.run(cmd,cwd=REPO,env=run_env,check=False)
backup_root=REPO/".colab_live_backup"; logs=sorted(backup_root.glob("*/train.log"),key=lambda p:p.stat().st_mtime) if backup_root.exists() else []
if logs: shutil.copy2(logs[-1],LOG)
if result.returncode: raise RuntimeError(f"trainer failed with exit code {result.returncode}; see {LOG}")
print("✅ trainer finished normally; needs_more_training is a scientific status, not a crash")

In [ ]:
def show(name):
 p=OUT/name
 if p.exists(): print(f"\n### {name}\n"+p.read_text())
for n in ["sequential_run_status.json","sequential_progress.json","sequential_in_progress.json","sequential_training_report.json"]: show(n)
progress_pt=OUT/"sequential_progress.pt"; accepted=[]
if progress_pt.exists():
 progress=torch.load(progress_pt,map_location="cpu",weights_only=False); accepted=[int(x) for x in progress.get("accepted_layers",[])]
print("Accepted:",accepted)

## Original vs accepted snapshot
Only accepted layers are loaded; an unaccepted current layer is excluded.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from tinycenn_lm.gemma3_memory_fusion import Gemma3MemoryFusionConfig, replace_attention_layers, structural_summary
D=torch.device("cuda"); DT=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
tok=AutoTokenizer.from_pretrained(MODEL_ID,revision=MODEL_REVISION)
baseline=AutoModelForCausalLM.from_pretrained(MODEL_ID,revision=MODEL_REVISION,torch_dtype=DT,attn_implementation="sdpa").to(D).eval()
student=AutoModelForCausalLM.from_pretrained(MODEL_ID,revision=MODEL_REVISION,torch_dtype=DT,attn_implementation="sdpa").to(D).eval()
progress=None; accepted=[]
if progress_pt.exists():
 progress=torch.load(progress_pt,map_location="cpu",weights_only=False); accepted=[int(x) for x in progress.get("accepted_layers",[])]; mf_cfg=Gemma3MemoryFusionConfig.from_dict(progress["config"])
 if accepted: replace_attention_layers(student,mf_cfg,accepted); student.load_state_dict(progress["attention_state"],strict=False)
print("Accepted",accepted); print(json.dumps(structural_summary(student),indent=2))
@torch.no_grad()
def gen(model,ids,n=40):
 s=ids.clone()
 for _ in range(n):
  x=model(input_ids=s,use_cache=False,return_dict=True).logits[:,-1].argmax(-1,keepdim=True); s=torch.cat([s,x],1)
  if tok.eos_token_id is not None and int(x)==int(tok.eos_token_id): break
 return tok.decode(s[0,ids.shape[1]:],skip_special_tokens=False)
tools=[{"type":"function","function":{"name":"get_current_temperature","description":"Get temperature for a city","parameters":{"type":"object","properties":{"location":{"type":"string"}},"required":["location"]}}},{"type":"function","function":{"name":"calculate","description":"Calculate expression","parameters":{"type":"object","properties":{"expression":{"type":"string"}},"required":["expression"]}}}]
comparisons=[]
for prompt in ["What's the temperature in Vienna?","Calculate 18 times 7.","What is the capital of Austria?"]:
 msgs=[{"role":"developer","content":"Use the available functions when needed."},{"role":"user","content":prompt}]
 ids=tok.apply_chat_template(msgs,tools=tools,add_generation_prompt=True,tokenize=True,return_tensors="pt").to(D)
 a,b=gen(baseline,ids),gen(student,ids); comparisons.append({"prompt":prompt,"original":a,"memory_fusion":b}); print("\n",prompt,"\nORIGINAL:",a,"\nMEMORY FUSION:",b)
(OUT/"prompt_comparison.json").write_text(json.dumps(comparisons,indent=2))

## Publish model/checkpoint to Hugging Face
If accepted layers exist, an inference adapter is published. The current unaccepted checkpoint is also copied under `training/` so it can be resumed.

In [ ]:
from tinycenn_lm.gemma3_memory_fusion import save_adapter
EXPORT=Path("/content/functiongemma-memory-fusion-hf")
if EXPORT.exists(): shutil.rmtree(EXPORT)
EXPORT.mkdir(); (EXPORT/"training").mkdir()
if accepted:
 save_adapter(student,EXPORT,config=mf_cfg,base_model=MODEL_ID,accepted_layers=accepted,metadata={"base_model_revision":MODEL_REVISION})
for n in ["sequential_run_status.json","sequential_progress.json","sequential_progress.pt","sequential_training_report.json","prompt_comparison.json"]:
 p=OUT/n
 if p.exists(): shutil.copy2(p,EXPORT/n)
for n in ["sequential_in_progress.json","sequential_in_progress.pt"]:
 p=OUT/n
 if p.exists(): shutil.copy2(p,EXPORT/"training"/n)
status=json.loads((OUT/"sequential_run_status.json").read_text()) if (OUT/"sequential_run_status.json").exists() else {}
rows=[]
if progress:
 for r in progress.get("layer_reports",[]): rows.append(f"| {r.get('layer')} | {r.get('accepted')} | {r.get('nmse',0):.5f} | {r.get('cosine',0):.5f} | {r.get('incremental_delta_nll',0):+.5f} |")
card_lines=[
 "---", "library_name: transformers", f"base_model: {MODEL_ID}",
 "tags: [gemma3, function-calling, tinycenn, recurrent-memory, research]", "---",
 "# FunctionGemma 270M + TinyCeNN Memory Fusion", "",
 f"Base revision: `{MODEL_REVISION}`  ", f"Accepted layers: `{accepted}`  ", f"Status: `{status.get('status','unknown')}`", "",
 "Only original full-attention anchors are replaced; Gemma3 sliding-window layers remain unchanged.", "",
 "| Layer | Accepted | NMSE | Cosine | Incremental ΔNLL |", "|---:|:---:|---:|---:|---:|",
 *(rows if rows else ["| — | — | — | — | — |"]), "",
 "Acceptance gates: NMSE ≤ 0.20, cosine ≥ 0.90, incremental ΔNLL ≤ +0.015, cumulative ΔNLL ≤ +0.05.",
 "`training/sequential_in_progress.pt` is resumable research state and may contain an unaccepted layer.", "",
 "Source: https://github.com/vtavakkoli/TinyCeNN-LM"
]
card="\n".join(card_lines)
(EXPORT/"README.md").write_text(card)
if PUBLISH_TO_HF:
 api=HfApi(); api.create_repo(HF_MODEL_REPO,repo_type="model",private=HF_PRIVATE,exist_ok=True); api.upload_folder(repo_id=HF_MODEL_REPO,repo_type="model",folder_path=str(EXPORT),commit_message=f"Memory Fusion update accepted={accepted} status={status.get('status','unknown')}")
 print("✅ Published:",f"https://huggingface.co/{HF_MODEL_REPO}")